# Pydantic model from text input

TODO: skapa en agent som ska simulera en it anställd

In [ ]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

employee_simultator = Agent("openrouter:openai/gpt-oss-120b:free",
system_prompt="""You are an HR expert within IT field in sweden within data science, data engineering,
machine learning, ai engineering. You will simulate AI employees

field to include in output:
- name
- age 
- gender
- job_title
- salary in SEK per month 
""")

result = await employee_simultator.run("simultate two employees")

print(result.output)

**Employee 1**

- **Name:** Anna Lindgren  
- **Age:** 32  
- **Gender:** Female  
- **Job Title:** Senior Data Engineer  
- **Salary:** 62 500 SEK per month  

**Employee 2**

- **Name:** Erik Svensson  
- **Age:** 28  
- **Gender:** Male  
- **Job Title:** Machine Learning Engineer  
- **Salary:** 55 000 SEK per month  


In [13]:
with open("simulated_employees", "w") as file:
    file.write(result.output)

## Get more structured output

issue with above:
- output structure vary
- hard to work with the data e.g. compute mean of salaries

want:
- repeatable structure

In [17]:
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent

class EmployeeModel(BaseModel):
    name: str = Field(description="Mostly swedish names, but could be foreign names as well")
    age: int = Field(description="age should be. between 18 and 67")
    gender: Literal["Male", "Female",]
    experience_level: Literal["Entry", "Mid level", "Senior", "Expert"]
    job_title: str
    salary: int = Field(
        gte=30_000, lte=50_000, description="salary should be between 30k and 50k, " \
        "the higher expernience level, the higher salary"
    )

employee_simultator = Agent("openrouter:openai/gpt-oss-120b:free",
system_prompt="""
You are an HR expert within IT field in sweden within data science, data engineering,
machine learning, ai engineering. You will simulate AI employees
""",
)

result = await employee_simultator.run("Give me 3 employees", output_type=EmployeeModel)
result

/var/folders/6x/30xtjdm94knggq9f67lxtmmh0000gn/T/ipykernel_2491/706936556.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(


AgentRunResult(output=EmployeeModel(name='Anna Svensson', age=29, gender='Female', experience_level='Mid level', job_title='Data Engineer', salary=38000))

In [18]:
result.output

EmployeeModel(name='Anna Svensson', age=29, gender='Female', experience_level='Mid level', job_title='Data Engineer', salary=38000)

In [20]:
result.output.salary+5000

43000

In [23]:
employee_simulator_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""", retries=1
)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=list[EmployeeModel], )
result

AgentRunResult(output=[EmployeeModel(name='Anna', age=25, gender='Female', experience_level='Mid level', job_title='Data Scientist', salary=40000), EmployeeModel(name='Erik', age=35, gender='Male', experience_level='Senior', job_title='Data Engineer', salary=45000), EmployeeModel(name='Sofia', age=40, gender='Female', experience_level='Expert', job_title='Machine Learning Engineer', salary=48000)])

In [ ]:
# Basemodel -> dictionary
result.output[0].model_dump()

{'name': 'Anna',
 'age': 25,
 'gender': 'Female',
 'experience_level': 'Mid level',
 'job_title': 'Data Scientist',
 'salary': 40000}

TODO: 
- Result.output make into list of dictionaries
- Create pandas dataframe based on this list
- export a csv file of our simulated employees

In [ ]:
import pandas as pd

list_of_employees = [employee.model_dump() for employee in result.output]

df = pd.DataFrame(list_of_employees)
df

,name,age,gender,experience_level,job_title,salary
0,Anna,25,Female,Mid level,Data Scientist,40000
1,Erik,35,Male,Senior,Data Engineer,45000
2,Sofia,40,Female,Expert,Machine Learning Engineer,48000


In [48]:
df["salary"].mean

<bound method Series.mean of 0    40000
1    45000
2    48000
Name: salary, dtype: int64>

In [51]:
df.to_csv("simulated_employees.csv", index=False)